# TrustPCB: Lightweight YOLO Baseline

## 1.0 Purpose

This notebook establishes the baseline PCB defect detector used in the TrustPCB research project.

The baseline provides a reference detector before introducing the reliability framework.

The main tasks are:

- verify the YOLO training environment,
- prepare the supplied and similarity-aware dataset configurations,
- train one lightweight YOLO detector,
- evaluate standard object-detection performance,
- record model size and inference efficiency, and
- prepare predictions for later reliability analysis.

The first baseline will use the supplied DsPCBSD+ split.

The similarity-aware split will later be used as an evaluation safeguard to check whether the identified cross-split similarities affect detector performance.

## 1.1 Environment Verification

The TrustPCB Python environment is checked before preparing the baseline experiment.

The following are verified:

- Python environment,
- PyTorch version,
- Ultralytics version,
- CUDA availability,
- GPU model, and
- project and dataset paths.

The baseline training will only begin after GPU access is confirmed.

In [2]:
# 1.1 Verify baseline environment

from pathlib import Path
import sys
import torch
import ultralytics

PROJECT_ROOT = Path("/scr/user/danielw9199/TrustPCB")

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DsPCBSD_plus"
    / "Data_YOLO"
)

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
CONFIG_DIR = PROJECT_ROOT / "configs"
RUNS_DIR = PROJECT_ROOT / "runs"

print("=== ENVIRONMENT ===")

print("Python:")
print(sys.executable)

print("\nPyTorch:")
print(torch.__version__)

print("\nUltralytics:")
print(ultralytics.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

    print("\nCUDA runtime:")
    print(torch.version.cuda)

print("\n=== PATHS ===")

for name, path in {
    "Project": PROJECT_ROOT,
    "Dataset": DATASET_ROOT,
    "Splits": SPLIT_DIR,
    "Configs": CONFIG_DIR,
    "Runs": RUNS_DIR,
}.items():
    print(f"{name:10s}: {path.exists()} | {path}")

=== ENVIRONMENT ===
Python:
/scr/user/danielw9199/TrustPCB/.conda/bin/python

PyTorch:
2.6.0+cu124

Ultralytics:
8.4.117

CUDA available:
True

GPU:
NVIDIA A100-SXM4-80GB

CUDA runtime:
12.4

=== PATHS ===
Project   : True | /scr/user/danielw9199/TrustPCB
Dataset   : True | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO
Splits    : True | /scr/user/danielw9199/TrustPCB/data/splits
Configs   : True | /scr/user/danielw9199/TrustPCB/configs
Runs      : True | /scr/user/danielw9199/TrustPCB/runs


## 1.2 Dataset and Split Verification

The DsPCBSD+ dataset and the supplied and similarity-aware split definitions are verified before baseline training.

This step confirms:

- the expected image and label directories,
- matching image-label counts,
- the expected training and validation split sizes,
- no overlap between training and validation assignments,
- use of the same complete image set, and
- the number of images reassigned by the similarity-aware split.

This ensures that the baseline experiments use a complete and consistent dataset before the YOLO configurations are prepared.

In [3]:
dataset_dirs = {
    "train_images": DATASET_ROOT / "images/train",
    "train_labels": DATASET_ROOT / "labels/train",
    "val_images": DATASET_ROOT / "images/val",
    "val_labels": DATASET_ROOT / "labels/val",
}

split_files = {
    "supplied_train": SPLIT_DIR / "supplied_train.txt",
    "supplied_val": SPLIT_DIR / "supplied_val.txt",
    "similarity_train": SPLIT_DIR / "similarity_aware_train.txt",
    "similarity_val": SPLIT_DIR / "similarity_aware_val.txt",
}

print("=== DATASET DIRECTORIES ===")

for name, path in dataset_dirs.items():
    count = sum(1 for p in path.iterdir() if p.is_file())
    print(f"{name:18s}: {count:5d} | {path}")


train_images = {p.stem for p in dataset_dirs["train_images"].glob("*.jpg")}
train_labels = {p.stem for p in dataset_dirs["train_labels"].glob("*.txt")}
val_images = {p.stem for p in dataset_dirs["val_images"].glob("*.jpg")}
val_labels = {p.stem for p in dataset_dirs["val_labels"].glob("*.txt")}

print("\n=== IMAGE-LABEL MATCHING ===")
print("Train images without labels:", len(train_images - train_labels))
print("Train labels without images:", len(train_labels - train_images))
print("Val images without labels  :", len(val_images - val_labels))
print("Val labels without images  :", len(val_labels - val_images))


splits = {}

for name, path in split_files.items():
    entries = [
        line.strip()
        for line in path.read_text().splitlines()
        if line.strip()
    ]
    splits[name] = entries

print("\n=== SPLIT COUNTS ===")

expected = {
    "supplied_train": 8208,
    "supplied_val": 2051,
    "similarity_train": 8207,
    "similarity_val": 2052,
}

for name, expected_count in expected.items():
    actual = len(splits[name])
    status = "OK" if actual == expected_count else "MISMATCH"
    print(f"{name:20s}: {actual:5d} | expected {expected_count:5d} | {status}")


supplied_train = set(splits["supplied_train"])
supplied_val = set(splits["supplied_val"])
similarity_train = set(splits["similarity_train"])
similarity_val = set(splits["similarity_val"])

supplied_all = supplied_train | supplied_val
similarity_all = similarity_train | similarity_val

print("\n=== SPLIT INTEGRITY ===")
print("Supplied train-val overlap        :", len(supplied_train & supplied_val))
print("Similarity-aware train-val overlap:", len(similarity_train & similarity_val))
print("Supplied total images             :", len(supplied_all))
print("Similarity-aware total images     :", len(similarity_all))
print("Same image universe               :", supplied_all == similarity_all)

changed = supplied_train ^ similarity_train

print("\n=== ASSIGNMENT CHANGES ===")
print("Moved into similarity-aware train:", len(similarity_train - supplied_train))
print("Moved into similarity-aware val  :", len(similarity_val - supplied_val))
print("Images with changed assignment   :", len(changed))

=== DATASET DIRECTORIES ===
train_images      :  8208 | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/images/train
train_labels      :  8208 | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/labels/train
val_images        :  2051 | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/images/val
val_labels        :  2051 | /scr/user/danielw9199/TrustPCB/data/raw/DsPCBSD_plus/Data_YOLO/labels/val

=== IMAGE-LABEL MATCHING ===
Train images without labels: 0
Train labels without images: 0
Val images without labels  : 0
Val labels without images  : 0

=== SPLIT COUNTS ===
supplied_train      :  8208 | expected  8208 | OK
supplied_val        :  2051 | expected  2051 | OK
similarity_train    :  8207 | expected  8207 | OK
similarity_val      :  2052 | expected  2052 | OK

=== SPLIT INTEGRITY ===
Supplied train-val overlap        : 0
Similarity-aware train-val overlap: 0
Supplied total images             : 10259
Similarity-aware total images     : 10259

## 1.3 Class Mapping Verification

The numerical YOLO class IDs are verified against the published DsPCBSD+ class statistics before baseline training.

The dataset paper defines nine PCB defect categories and reports the number of annotations for each category in the supplied training and validation sets.

The observed annotation counts from the local YOLO labels are compared with these published values to recover and verify the mapping between numerical class IDs and defect names.

In [4]:
from collections import Counter

train_label_dir = DATASET_ROOT / "labels/train"
val_label_dir = DATASET_ROOT / "labels/val"

def count_class_ids(label_dir):
    counts = Counter()

    for path in label_dir.glob("*.txt"):
        for line in path.read_text().splitlines():
            if line.strip():
                class_id = int(float(line.split()[0]))
                counts[class_id] += 1

    return counts


train_counts = count_class_ids(train_label_dir)
val_counts = count_class_ids(val_label_dir)

all_ids = sorted(set(train_counts) | set(val_counts))

print("=== OBSERVED YOLO CLASS COUNTS ===")

for class_id in all_ids:
    train = train_counts[class_id]
    val = val_counts[class_id]
    total = train + val

    print(
        f"ID {class_id}: "
        f"train={train:4d} | "
        f"val={val:4d} | "
        f"total={total:4d}"
    )


published_counts = {
    "SH":   (746, 169, 915),
    "SP":   (3655, 929, 4584),
    "SC":   (1308, 285, 1593),
    "OP":   (1432, 338, 1770),
    "MB":   (1983, 546, 2529),
    "HB":   (2275, 608, 2883),
    "CS":   (2042, 448, 2490),
    "CFO":  (1409, 423, 1832),
    "BMFO": (1334, 346, 1680),
}

print("\n=== VERIFIED CLASS MAPPING ===")

class_mapping = {}

for class_id in all_ids:
    observed = (
        train_counts[class_id],
        val_counts[class_id],
        train_counts[class_id] + val_counts[class_id],
    )

    matches = [
        name
        for name, expected in published_counts.items()
        if observed == expected
    ]

    if len(matches) == 1:
        class_mapping[class_id] = matches[0]
        print(f"{class_id} -> {matches[0]} | counts match publication")
    else:
        print(f"{class_id} -> UNRESOLVED | observed={observed}")


print("\n=== TOTAL ANNOTATIONS ===")
print("Training  :", sum(train_counts.values()), "| expected: 16184")
print("Validation:", sum(val_counts.values()), "| expected: 4092")
print(
    "Total     :",
    sum(train_counts.values()) + sum(val_counts.values()),
    "| expected: 20276"
)

=== OBSERVED YOLO CLASS COUNTS ===
ID 0: train= 746 | val= 169 | total= 915
ID 1: train=3655 | val= 929 | total=4584
ID 2: train=1308 | val= 285 | total=1593
ID 3: train=1432 | val= 338 | total=1770
ID 4: train=1983 | val= 546 | total=2529
ID 5: train=2275 | val= 608 | total=2883
ID 6: train=2042 | val= 448 | total=2490
ID 7: train=1409 | val= 423 | total=1832
ID 8: train=1334 | val= 346 | total=1680

=== VERIFIED CLASS MAPPING ===
0 -> SH | counts match publication
1 -> SP | counts match publication
2 -> SC | counts match publication
3 -> OP | counts match publication
4 -> MB | counts match publication
5 -> HB | counts match publication
6 -> CS | counts match publication
7 -> CFO | counts match publication
8 -> BMFO | counts match publication

=== TOTAL ANNOTATIONS ===
Training  : 16184 | expected: 16184
Validation: 4092 | expected: 4092
Total     : 20276 | expected: 20276


## 1.4 YOLO Dataset Configuration

YOLO dataset configurations are prepared for the supplied and similarity-aware split schemes.

The original DsPCBSD+ image and label directories remain unchanged. Each experimental split is represented using image-path lists so that alternative train-validation assignments can be evaluated without moving or duplicating the raw dataset.

This step:

- generates image-path lists for both split schemes,
- creates the corresponding YOLO dataset configuration files, and
- validates the image paths, annotations, split sizes, and class definitions.

Both configurations use the same 10,259 images and the same nine verified defect classes. Only the training and validation assignments differ.

In [5]:
import yaml

DATASET_CONFIG_DIR = CONFIG_DIR / "datasets"
DATASET_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

image_dirs = [
    DATASET_ROOT / "images/train",
    DATASET_ROOT / "images/val",
]

image_index = {}

for image_dir in image_dirs:
    for path in image_dir.glob("*.jpg"):
        if path.name in image_index:
            raise ValueError(f"Duplicate image filename: {path.name}")
        image_index[path.name] = path.resolve()


def resolve_split(entries):
    paths = []

    for filename in entries:
        if filename not in image_index:
            raise FileNotFoundError(f"Image not found: {filename}")
        paths.append(image_index[filename])

    return paths


resolved_splits = {
    name: resolve_split(entries)
    for name, entries in splits.items()
}

class_names = {
    0: "SH",
    1: "SP",
    2: "SC",
    3: "OP",
    4: "MB",
    5: "HB",
    6: "CS",
    7: "CFO",
    8: "BMFO",
}

list_files = {
    "supplied_train": DATASET_CONFIG_DIR / "supplied_train_images.txt",
    "supplied_val": DATASET_CONFIG_DIR / "supplied_val_images.txt",
    "similarity_train": DATASET_CONFIG_DIR / "similarity_aware_train_images.txt",
    "similarity_val": DATASET_CONFIG_DIR / "similarity_aware_val_images.txt",
}

for name, output_path in list_files.items():
    output_path.write_text(
        "\n".join(str(path) for path in resolved_splits[name]) + "\n"
    )

dataset_configs = {
    "supplied": {
        "train": str(list_files["supplied_train"].resolve()),
        "val": str(list_files["supplied_val"].resolve()),
        "names": class_names,
    },
    "similarity_aware": {
        "train": str(list_files["similarity_train"].resolve()),
        "val": str(list_files["similarity_val"].resolve()),
        "names": class_names,
    },
}

yaml_files = {
    "supplied": DATASET_CONFIG_DIR / "dspcbsd_supplied.yaml",
    "similarity_aware": DATASET_CONFIG_DIR / "dspcbsd_similarity_aware.yaml",
}

for name, config in dataset_configs.items():
    yaml_files[name].write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

print("=== GENERATED DATASET CONFIGURATIONS ===")

for name, path in yaml_files.items():
    print(f"\n{name}:")
    print(path)
    print(path.read_text())

=== GENERATED DATASET CONFIGURATIONS ===

supplied:
/scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_supplied.yaml
train: /scr/user/danielw9199/TrustPCB/configs/datasets/supplied_train_images.txt
val: /scr/user/danielw9199/TrustPCB/configs/datasets/supplied_val_images.txt
names:
  0: SH
  1: SP
  2: SC
  3: OP
  4: MB
  5: HB
  6: CS
  7: CFO
  8: BMFO


similarity_aware:
/scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_similarity_aware.yaml
train: /scr/user/danielw9199/TrustPCB/configs/datasets/similarity_aware_train_images.txt
val: /scr/user/danielw9199/TrustPCB/configs/datasets/similarity_aware_val_images.txt
names:
  0: SH
  1: SP
  2: SC
  3: OP
  4: MB
  5: HB
  6: CS
  7: CFO
  8: BMFO



In [6]:
from ultralytics.data.utils import check_det_dataset

config_specs = {
    "supplied": {
        "yaml": DATASET_CONFIG_DIR / "dspcbsd_supplied.yaml",
        "train": DATASET_CONFIG_DIR / "supplied_train_images.txt",
        "val": DATASET_CONFIG_DIR / "supplied_val_images.txt",
        "expected_train": 8208,
        "expected_val": 2051,
    },
    "similarity_aware": {
        "yaml": DATASET_CONFIG_DIR / "dspcbsd_similarity_aware.yaml",
        "train": DATASET_CONFIG_DIR / "similarity_aware_train_images.txt",
        "val": DATASET_CONFIG_DIR / "similarity_aware_val_images.txt",
        "expected_train": 8207,
        "expected_val": 2052,
    },
}


def read_image_paths(path):
    return [
        Path(line.strip())
        for line in path.read_text().splitlines()
        if line.strip()
    ]


def get_label_path(image_path):
    parts = list(image_path.parts)
    index = parts.index("images")
    parts[index] = "labels"
    return Path(*parts).with_suffix(".txt")


print("=== DATASET CONFIGURATION VALIDATION ===")

for name, spec in config_specs.items():
    train_paths = read_image_paths(spec["train"])
    val_paths = read_image_paths(spec["val"])
    all_paths = train_paths + val_paths

    missing_images = [p for p in all_paths if not p.exists()]
    missing_labels = [
        get_label_path(p)
        for p in all_paths
        if not get_label_path(p).exists()
    ]

    data = check_det_dataset(str(spec["yaml"]))

    loaded_names = data["names"]
    if isinstance(loaded_names, list):
        loaded_names = {i: value for i, value in enumerate(loaded_names)}
    else:
        loaded_names = {int(k): v for k, v in loaded_names.items()}

    print(f"\n--- {name.upper()} ---")
    print("Train images      :", len(train_paths), "| expected:", spec["expected_train"])
    print("Val images        :", len(val_paths), "| expected:", spec["expected_val"])
    print("Total images      :", len(all_paths))
    print("Duplicate train   :", len(train_paths) - len(set(train_paths)))
    print("Duplicate val     :", len(val_paths) - len(set(val_paths)))
    print("Missing images    :", len(missing_images))
    print("Missing labels    :", len(missing_labels))
    print("Classes           :", data["nc"])
    print("Class names       :", loaded_names)

    assert len(train_paths) == spec["expected_train"]
    assert len(val_paths) == spec["expected_val"]
    assert len(all_paths) == 10259
    assert len(train_paths) == len(set(train_paths))
    assert len(val_paths) == len(set(val_paths))
    assert not missing_images
    assert not missing_labels
    assert data["nc"] == 9
    assert loaded_names == class_names

    print("Validation status : PASSED")

=== DATASET CONFIGURATION VALIDATION ===

--- SUPPLIED ---
Train images      : 8208 | expected: 8208
Val images        : 2051 | expected: 2051
Total images      : 10259
Duplicate train   : 0
Duplicate val     : 0
Missing images    : 0
Missing labels    : 0
Classes           : 9
Class names       : {0: 'SH', 1: 'SP', 2: 'SC', 3: 'OP', 4: 'MB', 5: 'HB', 6: 'CS', 7: 'CFO', 8: 'BMFO'}
Validation status : PASSED

--- SIMILARITY_AWARE ---
Train images      : 8207 | expected: 8207
Val images        : 2052 | expected: 2052
Total images      : 10259
Duplicate train   : 0
Duplicate val     : 0
Missing images    : 0
Missing labels    : 0
Classes           : 9
Class names       : {0: 'SH', 1: 'SP', 2: 'SC', 3: 'OP', 4: 'MB', 5: 'HB', 6: 'CS', 7: 'CFO', 8: 'BMFO'}
Validation status : PASSED


# 2.0 Preliminary Baseline Experiment

The preliminary baseline experiment establishes a lightweight reference detector for TrustPCB.

A pretrained YOLOv8n detector is used as the fixed baseline model. The purpose is not to modify the detector architecture, but to establish standard object-detection performance before the reliability framework is introduced.

The experiment first performs a short smoke test to verify the complete training pipeline. Two matched baseline runs are then conducted using:

- the supplied DsPCBSD+ split, and
- the similarity-aware split.

Both runs use the same model, training settings, random seed, and software environment so that the dataset split is the primary experimental difference.

## 2.1 Baseline Configuration

The preliminary baseline training configuration is fixed before any model training is performed.

The experiment uses:

- YOLOv8n with pretrained weights,
- an input image size of 640 × 640,
- 100 training epochs,
- a fixed random seed of 24209199,
- deterministic training where supported,
- a fixed batch size,
- standard Ultralytics training augmentation, and
- the same configuration for both dataset splits.

Ultralytics training outputs and run arguments are retained so that the experiment can be reproduced and compared consistently.

In [7]:
import os
import random
import numpy as np
import torch

EXPERIMENT_VERSION = "v1"
EXPERIMENT_SEED = 24209199

EXPERIMENT_CONFIG_DIR = CONFIG_DIR / "experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

VERSION_RUN_DIR = (
    RUNS_DIR
    / "preliminary_baseline"
    / EXPERIMENT_VERSION
)

os.environ["PYTHONHASHSEED"] = str(EXPERIMENT_SEED)

random.seed(EXPERIMENT_SEED)
np.random.seed(EXPERIMENT_SEED)

torch.manual_seed(EXPERIMENT_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(EXPERIMENT_SEED)
    torch.cuda.manual_seed_all(EXPERIMENT_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASELINE_CONFIG = {
    "model": "yolov8n.pt",
    "imgsz": 640,
    "epochs": 100,
    "batch": 32,
    "seed": EXPERIMENT_SEED,
    "deterministic": True,
    "device": 0,
    "workers": 2,
    "patience": 100,
    "pretrained": True,
    "amp": True,
    "plots": True,
    "save": True,
    "val": True,
}

BASELINE_CONFIG_PATH = (
    EXPERIMENT_CONFIG_DIR
    / f"yolov8n_preliminary_baseline_{EXPERIMENT_VERSION}.yaml"
)

BASELINE_CONFIG_PATH.write_text(
    yaml.safe_dump(BASELINE_CONFIG, sort_keys=False)
)

print("=== REPRODUCIBILITY CONTROL ===")
print("Version            :", EXPERIMENT_VERSION)
print("Experimental seed  :", EXPERIMENT_SEED)
print("Python hash seed   :", os.environ["PYTHONHASHSEED"])
print("NumPy seed         : set")
print("PyTorch seed       :", torch.initial_seed())
print("CUDA available     :", torch.cuda.is_available())
print("cuDNN deterministic:", torch.backends.cudnn.deterministic)
print("cuDNN benchmark    :", torch.backends.cudnn.benchmark)

print("\n=== PRELIMINARY BASELINE CONFIGURATION ===")
print(BASELINE_CONFIG_PATH)
print()
print(BASELINE_CONFIG_PATH.read_text())

=== REPRODUCIBILITY CONTROL ===
Version            : v1
Experimental seed  : 24209199
Python hash seed   : 24209199
NumPy seed         : set
PyTorch seed       : 24209199
CUDA available     : True
cuDNN deterministic: True
cuDNN benchmark    : False

=== PRELIMINARY BASELINE CONFIGURATION ===
/scr/user/danielw9199/TrustPCB/configs/experiments/yolov8n_preliminary_baseline_v1.yaml

model: yolov8n.pt
imgsz: 640
epochs: 100
batch: 32
seed: 24209199
deterministic: true
device: 0
workers: 2
patience: 100
pretrained: true
amp: true
plots: true
save: true
val: true



## 2.2 Smoke Test

A short training run is performed on the supplied DsPCBSD+ split before the full baseline experiment.

The smoke test uses the same model and core training configuration as the preliminary baseline, but training is limited to two epochs.

The purpose is to verify that:

- the pretrained YOLOv8n model loads correctly,
- the dataset and annotations are accepted by Ultralytics,
- GPU training operates without errors,
- training and validation complete successfully, and
- experiment outputs are saved to the expected project directory.

The smoke-test metrics are used only for pipeline verification and are not treated as baseline results.

In [7]:
from ultralytics import YOLO

SMOKE_RUN_NAME = "smoke_test"
smoke_run_path = VERSION_RUN_DIR / SMOKE_RUN_NAME

if smoke_run_path.exists():
    raise FileExistsError(
        f"Run directory already exists: {smoke_run_path}"
    )

smoke_config = {
    key: value
    for key, value in BASELINE_CONFIG.items()
    if key not in {"model", "epochs"}
}

smoke_config.update({
    "data": str(
        DATASET_CONFIG_DIR
        / "dspcbsd_supplied.yaml"
    ),
    "epochs": 2,
    "project": str(VERSION_RUN_DIR),
    "name": SMOKE_RUN_NAME,
    "exist_ok": False,
})

assert EXPERIMENT_SEED == 24209199
assert BASELINE_CONFIG["seed"] == EXPERIMENT_SEED
assert smoke_config["seed"] == EXPERIMENT_SEED

print("=== SMOKE TEST ===")
print("Version:", EXPERIMENT_VERSION)
print("Model  :", BASELINE_CONFIG["model"])
print("Data   :", smoke_config["data"])
print("Epochs :", smoke_config["epochs"])
print("Batch  :", smoke_config["batch"])
print("Workers:", smoke_config["workers"])
print("Seed   :", smoke_config["seed"])
print("Output :", smoke_run_path)

smoke_model = YOLO(BASELINE_CONFIG["model"])
smoke_results = smoke_model.train(**smoke_config)

=== SMOKE TEST ===
Version: v1
Model  : yolov8n.pt
Data   : /scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_supplied.yaml
Epochs : 2
Batch  : 32
Workers: 2
Seed   : 24209199
Output : /scr/user/danielw9199/TrustPCB/runs/preliminary_baseline/v1/smoke_test
New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_supplied.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, 

## 2.3 Supplied-Split Baseline

The first full preliminary baseline is trained using the original supplied DsPCBSD+ train-validation split.

YOLOv8n is trained for 100 epochs using the frozen baseline configuration established in Section 2.1.

This run provides:

- standard object-detection performance on the supplied split,
- per-class detection behaviour,
- training and validation learning curves,
- model weights and experiment metadata, and
- the reference result for the later comparison with the similarity-aware split.

The resulting metrics are treated as preliminary baseline results for TrustPCB.

In [ ]:
from ultralytics import YOLO

SUPPLIED_RUN_NAME = "supplied"
supplied_run_path = VERSION_RUN_DIR / SUPPLIED_RUN_NAME

if supplied_run_path.exists():
    raise FileExistsError(
        f"Run directory already exists: {supplied_run_path}"
    )

supplied_config = {
    key: value
    for key, value in BASELINE_CONFIG.items()
    if key != "model"
}

supplied_config.update({
    "data": str(DATASET_CONFIG_DIR / "dspcbsd_supplied.yaml"),
    "project": str(VERSION_RUN_DIR),
    "name": SUPPLIED_RUN_NAME,
    "exist_ok": False,
})

assert EXPERIMENT_SEED == 24209199
assert BASELINE_CONFIG["seed"] == 24209199
assert supplied_config["seed"] == 24209199

print("=== SUPPLIED-SPLIT BASELINE ===")
print("Version:", EXPERIMENT_VERSION)
print("Model  :", BASELINE_CONFIG["model"])
print("Data   :", supplied_config["data"])
print("Epochs :", supplied_config["epochs"])
print("Batch  :", supplied_config["batch"])
print("Image  :", supplied_config["imgsz"])
print("Workers:", supplied_config["workers"])
print("Seed   :", supplied_config["seed"])
print("Output :", supplied_run_path)

supplied_model = YOLO(BASELINE_CONFIG["model"])
supplied_results = supplied_model.train(**supplied_config)

=== SUPPLIED-SPLIT BASELINE ===
Version: v1
Model  : yolov8n.pt
Data   : /scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_supplied.yaml
Epochs : 100
Batch  : 32
Image  : 640
Workers: 2
Seed   : 24209199
Output : /scr/user/danielw9199/TrustPCB/runs/preliminary_baseline/v1/supplied
New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_supplied.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0

## 2.4 Similarity-Aware Baseline

The second full preliminary baseline is trained using the similarity-aware DsPCBSD+ train-validation split established during the dataset audit.

The detector architecture, training configuration, random seed, and software environment are kept identical to the supplied-split baseline. The dataset split is therefore the primary experimental difference between the two runs.

This experiment evaluates whether the limited cross-split near-duplicate groups identified in the supplied DsPCBSD+ split materially affect baseline detection performance.

The results are compared with the supplied-split baseline in Section 2.5.

In [8]:
from ultralytics import YOLO

SIMILARITY_RUN_NAME = "similarity_aware"
similarity_run_path = VERSION_RUN_DIR / SIMILARITY_RUN_NAME

if similarity_run_path.exists():
    raise FileExistsError(
        f"Run directory already exists: {similarity_run_path}"
    )

similarity_config = {
    key: value
    for key, value in BASELINE_CONFIG.items()
    if key != "model"
}

similarity_config.update({
    "data": str(
        DATASET_CONFIG_DIR
        / "dspcbsd_similarity_aware.yaml"
    ),
    "project": str(VERSION_RUN_DIR),
    "name": SIMILARITY_RUN_NAME,
    "exist_ok": False,
})

assert EXPERIMENT_SEED == 24209199
assert BASELINE_CONFIG["seed"] == 24209199
assert similarity_config["seed"] == 24209199

print("=== SIMILARITY-AWARE BASELINE ===")
print("Version:", EXPERIMENT_VERSION)
print("Model  :", BASELINE_CONFIG["model"])
print("Data   :", similarity_config["data"])
print("Epochs :", similarity_config["epochs"])
print("Batch  :", similarity_config["batch"])
print("Image  :", similarity_config["imgsz"])
print("Workers:", similarity_config["workers"])
print("Seed   :", similarity_config["seed"])
print("Output :", similarity_run_path)

similarity_model = YOLO(BASELINE_CONFIG["model"])
similarity_results = similarity_model.train(**similarity_config)

=== SIMILARITY-AWARE BASELINE ===
Version: v1
Model  : yolov8n.pt
Data   : /scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_similarity_aware.yaml
Epochs : 100
Batch  : 32
Image  : 640
Workers: 2
Seed   : 24209199
Output : /scr/user/danielw9199/TrustPCB/runs/preliminary_baseline/v1/similarity_aware
New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/scr/user/danielw9199/TrustPCB/configs/datasets/dspcbsd_similarity_aware.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_mode

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



## 2.5 Baseline Comparison

The supplied-split and similarity-aware YOLOv8n baselines are compared using their best validation checkpoints, selected according to mAP@0.5:0.95.

Both experiments use the same detector architecture, training configuration, random seed, and software environment. The dataset split is the primary experimental difference.

The comparison focuses on precision, recall, mAP@0.5, and mAP@0.5:0.95 to assess whether the limited cross-split image similarity identified during the dataset audit materially affects aggregate baseline performance.

This comparison is treated as preliminary single-seed evidence for the RQ1 evaluation safeguard rather than as a final repeated-run statistical conclusion.

In [10]:
import pandas as pd
from pathlib import Path

supplied_results_path = (
    VERSION_RUN_DIR
    / "supplied"
    / "results.csv"
)

similarity_results_path = (
    VERSION_RUN_DIR
    / "similarity_aware"
    / "results.csv"
)

supplied_df = pd.read_csv(supplied_results_path)
similarity_df = pd.read_csv(similarity_results_path)

supplied_df.columns = supplied_df.columns.str.strip()
similarity_df.columns = similarity_df.columns.str.strip()

metric_columns = {
    "Precision": "metrics/precision(B)",
    "Recall": "metrics/recall(B)",
    "mAP50": "metrics/mAP50(B)",
    "mAP50-95": "metrics/mAP50-95(B)",
}

supplied_best_idx = supplied_df[
    metric_columns["mAP50-95"]
].idxmax()

similarity_best_idx = similarity_df[
    metric_columns["mAP50-95"]
].idxmax()

supplied_best = supplied_df.loc[supplied_best_idx]
similarity_best = similarity_df.loc[similarity_best_idx]

comparison = pd.DataFrame({
    "Metric": [
        "Best epoch",
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95",
    ],
    "Supplied": [
        int(supplied_best["epoch"]),
        supplied_best[metric_columns["Precision"]],
        supplied_best[metric_columns["Recall"]],
        supplied_best[metric_columns["mAP50"]],
        supplied_best[metric_columns["mAP50-95"]],
    ],
    "Similarity-aware": [
        int(similarity_best["epoch"]),
        similarity_best[metric_columns["Precision"]],
        similarity_best[metric_columns["Recall"]],
        similarity_best[metric_columns["mAP50"]],
        similarity_best[metric_columns["mAP50-95"]],
    ],
})

comparison["Difference"] = (
    comparison["Similarity-aware"]
    - comparison["Supplied"]
)

comparison.loc[
    comparison["Metric"] == "Best epoch",
    "Difference"
] = pd.NA

print("=== 2.5 PRELIMINARY BASELINE COMPARISON ===")
print(comparison.to_string(index=False))

PROJECT_ROOT = VERSION_RUN_DIR.parents[2]

OUTPUT_TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

OUTPUT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    OUTPUT_TABLE_DIR
    / "preliminary_baseline_comparison_v1.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

print("\nSaved comparison:")
print(comparison_path)

=== 2.5 PRELIMINARY BASELINE COMPARISON ===
    Metric  Supplied  Similarity-aware  Difference
Best epoch  76.00000          81.00000         NaN
 Precision   0.81872           0.81953     0.00081
    Recall   0.79505           0.80172     0.00667
     mAP50   0.84516           0.84864     0.00348
  mAP50-95   0.50529           0.50841     0.00312

Saved comparison:
/scr/user/danielw9199/TrustPCB/outputs/tables/preliminary_baseline_comparison_v1.csv


### Preliminary Comparison Result

The similarity-aware split produced performance comparable to the supplied DsPCBSD+ split. The similarity-aware baseline achieved slightly higher recall, mAP@0.5, and mAP@0.5:0.95, although the absolute differences were small.

The similarity-aware run achieved an mAP@0.5 of 0.8486 and an mAP@0.5:0.95 of 0.5084, compared with 0.8452 and 0.5053, respectively, for the supplied split.

These preliminary single-seed results suggest that the limited confirmed cross-split near-duplicate groups identified during the dataset audit do not materially affect aggregate YOLOv8n baseline performance under the current experimental setting. The result supports the use of similarity-aware splitting as an evaluation safeguard while avoiding a claim that the supplied split is free from all forms of data leakage.